# Lollipop chart for Campaigns 1–6 (WCM Basis)

This notebook loads the campaign summary data (computed on a **Wet Cell Matter (WCM)** basis) and creates one overlaid lollipop chart per KPI for interactive exploration. Each operating mode is shown on the y-axis; campaigns are offset slightly within each mode so their min–max ranges and mean markers remain readable.

In [4]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

ROOT = Path.cwd().parent
if not (ROOT / "Campaign_1_fixed_stoichiometry_Humbird").exists():
    ROOT = Path(r"C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code")

CROSS_CAMPAIGN_DIR = ROOT / "Cross_Campaign_Comparison"

# Use full summary_comparison.csv per campaign to access all WCM metrics directly
CAMPAIGN_SUMMARY_CSVS = {
    "Campaign_1": ROOT / "Campaign_1_fixed_stoichiometry_Humbird" / "06_Analysis" / "summary_comparison.csv",
    "Campaign_6": ROOT / "Campaign_6_fixed_stoichiometry_Humbird_enhanced" / "06_Analysis" / "summary_comparison.csv",
    "Campaign_2": ROOT / "Campaign_2_fixed_stoichiometry_GFI" / "06_Analysis" / "summary_comparison.csv",
    # "Campaign_3": ROOT / "Campaign_3_dynamic_stoichiometry_GFI" / "06_Analysis" / "summary_comparison.csv",
    "Campaign_4": ROOT / "Campaign_4_mode_comparison_excesskinetics" / "06_Analysis" / "summary_comparison.csv",
    "Campaign_5": ROOT / "Campaign_5_mode_comparison_adjustedexcesskinetics" / "06_Analysis" / "summary_comparison.csv",
}

KPI_ORDER = [
    "STY_WCM_kg_m3_day",
    "total_WCM_harvested_kg",
    "medium_conversion_efficiency_m3_kg_WCM",
    "feed_conversion_efficiency_kg_kg_WCM",
    "protein_conversion_efficiency_kg_kg",
]

KPI_LABELS = {
    "STY_WCM_kg_m3_day": "STY<sub>WCM</sub> (kg<sub>WCM</sub> m⁻³ day⁻¹)",
    "total_WCM_harvested_kg": "Harvested Wet Cell Mass (kg<sub>WCM</sub>)",
    "medium_conversion_efficiency_m3_kg_WCM": "1/MCE (kg<sub>WCM</sub> m⁻³)",
    "feed_conversion_efficiency_kg_kg_WCM": "1/FCE (kg<sub>WCM</sub> kg<sub>Nutrients</sub>⁻¹)",
    "protein_conversion_efficiency_kg_kg": "1/PCE (kg<sub>protein</sub> kg<sub>AA</sub>⁻¹)",
}

# Lower conversion efficiency is preferable, so reciprocal values are plotted
# to make higher values consistently represent better performance.
INVERSE_CONVERSION_KPIS = {
    "medium_conversion_efficiency_m3_kg_WCM",
    "feed_conversion_efficiency_kg_kg_WCM",
    "protein_conversion_efficiency_kg_kg",
}

# Conventional meat production benchmarks (raw literature values on wet weight basis, matching ParetoFront)
MEAT_BENCHMARKS_RAW = {
    "Beef": {
        "STY_WCM_kg_m3_day": 0.1151,
        "medium_conversion_efficiency_m3_kg_WCM": 0.2608,
        "protein_conversion_efficiency_kg_kg": 28.80,
        "feed_conversion_efficiency_kg_kg_WCM": 36.00,
    },
    "Pork": {
        "STY_WCM_kg_m3_day": 0.3381,
        "medium_conversion_efficiency_m3_kg_WCM": 0.1619,
        "protein_conversion_efficiency_kg_kg": 7.29,
        "feed_conversion_efficiency_kg_kg_WCM": 6.00,
    },
    "Poultry": {
        "STY_WCM_kg_m3_day": 0.5398,
        "medium_conversion_efficiency_m3_kg_WCM": 0.0631,
        "protein_conversion_efficiency_kg_kg": 3.57,
        "feed_conversion_efficiency_kg_kg_WCM": 4.20,
    },
}

# Plotted values (reciprocal for inverse conversion efficiencies)
MEAT_BENCHMARKS_PLOTTED = {
    meat: {
        kpi: (1.0 / val if kpi in INVERSE_CONVERSION_KPIS and val is not None and val != 0 else val)
        for kpi, val in metrics.items()
    }
    for meat, metrics in MEAT_BENCHMARKS_RAW.items()
}

MODE_ORDER = [
    "Batch",
    "FedBatch",
    "Chemostat",
    "Concentrated FedBatch",
    "Continuous Perfusion",
]

CAMPAIGN_LABELS = {
    "Campaign_1": "1. Fixed - Humbird (2020) Wildtype",
    "Campaign_2": "3. Fixed - Frohlich et al. (2025)",
    # "Campaign_3": "4. Growth rate dependent",
    "Campaign_4": "4. Substrate Dependent - Frohlich et al. (2025)",
    "Campaign_5": "5. Substrate Dependent Enhanced - Frohlich et al. (2025)",
    "Campaign_6": "2. Fixed - Humbird (2020) Enhanced",
}

CAMPAIGN_COLORS = {
    "Campaign_1": "#56B4E9",
    "Campaign_2": "#0072B2",
    # "Campaign_3": "#D55E00",
    "Campaign_4": "#009E73",
    "Campaign_5": "#CC79A7",
    "Campaign_6": "#E69F00",
}

CAMPAIGN_SYMBOLS = {
    "Campaign_1": "circle",
    "Campaign_2": "square",
    # "Campaign_3": "diamond",
    "Campaign_4": "triangle-up",
    "Campaign_5": "triangle-down",
    "Campaign_6": "hexagon",
}


def normalize_mode(value: object) -> str:
    mode = re.sub(r"^\d+_", "", str(value)).replace("_", " ")
    return {
        "Batch": "Batch",
        "FedBatch": "FedBatch",
        "Chemostat": "Chemostat",
        "ConcentratedFedBatch": "Concentrated FedBatch",
        "ContinuousPerfusion": "Continuous Perfusion",
    }.get(mode, mode)


frames = []
missing_files = []
for campaign, csv_path in CAMPAIGN_SUMMARY_CSVS.items():
    if not csv_path.exists():
        missing_files.append(str(csv_path))
        continue
    df = pd.read_csv(csv_path)
    df["campaign"] = campaign
    df["mode_group"] = df["mode"].map(normalize_mode)

    present_kpis = [k for k in KPI_ORDER if k in df.columns]
    for kpi in present_kpis:
        df[kpi] = pd.to_numeric(df[kpi], errors="coerce")

    grouped = df.groupby(["campaign", "mode_group"], observed=False)
    for kpi in present_kpis:
        agg = grouped[kpi].agg(["min", "max", "mean", "count"]).reset_index()
        agg = agg.rename(
            columns={
                "min": "min_value",
                "max": "max_value",
                "mean": "mean_value",
                "count": "n_runs",
            }
        )
        agg["kpi"] = kpi
        frames.append(agg)

if missing_files:
    raise FileNotFoundError(
        "Missing campaign KPI summary file(s): " + ", ".join(missing_files)
    )

summary_all = pd.concat(frames, ignore_index=True)
summary_all["min_value"] = pd.to_numeric(summary_all["min_value"], errors="coerce")
summary_all["max_value"] = pd.to_numeric(summary_all["max_value"], errors="coerce")
summary_all["mean_value"] = pd.to_numeric(summary_all["mean_value"], errors="coerce")
summary_all = summary_all.dropna(subset=["min_value", "max_value", "mean_value"])
summary_all = summary_all[summary_all["kpi"].isin(KPI_ORDER)].copy()

inverse_mask = summary_all["kpi"].isin(INVERSE_CONVERSION_KPIS)
inverse_values = summary_all.loc[inverse_mask, ["min_value", "max_value", "mean_value"]]
valid_inverse_mask = (inverse_values > 0).all(axis=1)
summary_all = summary_all.loc[~inverse_mask | valid_inverse_mask].copy()
summary_all.loc[inverse_mask & valid_inverse_mask, ["min_value", "max_value", "mean_value"]] = (
    1 / summary_all.loc[
        inverse_mask & valid_inverse_mask,
        ["max_value", "min_value", "mean_value"],
    ].to_numpy()
)

summary_all["mode_group"] = pd.Categorical(
    summary_all["mode_group"], categories=MODE_ORDER, ordered=True
)
summary_all = summary_all.sort_values(["kpi", "mode_group", "campaign"])

available_modes = set(summary_all["mode_group"].dropna().astype(str))
missing_modes = [mode for mode in MODE_ORDER if mode not in available_modes]
print(f"Loaded {len(summary_all)} campaign-mode-KPI summaries.")
if missing_modes:
    print(
        "No source summary rows are available for: "
        + ", ".join(missing_modes)
        + ". These modes will remain empty until their campaign summary tables are regenerated."
    )
display(summary_all[["campaign", "mode_group", "kpi", "min_value", "max_value", "mean_value", "n_runs"]].head())

Loaded 125 campaign-mode-KPI summaries.


,campaign,mode_group,kpi,min_value,max_value,mean_value,n_runs
0,Campaign_1,Batch,STY_WCM_kg_m3_day,1.413807,1.413807,1.413807,1
50,Campaign_2,Batch,STY_WCM_kg_m3_day,3.954041,3.954041,3.954041,1
75,Campaign_4,Batch,STY_WCM_kg_m3_day,1.019839,1.019839,1.019839,1
100,Campaign_5,Batch,STY_WCM_kg_m3_day,4.467764,4.467764,4.467764,1
25,Campaign_6,Batch,STY_WCM_kg_m3_day,7.617561,7.617561,7.617561,1


In [5]:
# One subplot per KPI; each row contains min–max stems and a mean marker for each campaign.
def create_lollipop_figure(
    selected_campaigns,
    *,
    selected_kpis=None,
    height_per_kpi=350,
    width=1100,
    show_reference_bands=True,
    row_step=1.5,
):
    """Create an interactive multi-KPI lollipop figure for selected campaigns.

    Parameters
    ----------
    selected_campaigns : list of str
        List of campaign identifiers to include.
    selected_kpis : list of str, optional
        List of KPI identifiers to include. Defaults to all KPIs in KPI_ORDER.
    height_per_kpi : int, default 420
        Height in pixels allocated per KPI subplot row.
    width : int, default 1100
        Total width of the figure in pixels.
    show_reference_bands : bool, default True
        Whether to draw shaded reference spans representing conventional meat (Beef, Pork, Poultry).
    row_step : float, default 1.0
        Distance between operational mode row centers to provide vertical spacing.
    """
    # =========================================================================
    # 1. INPUT VALIDATION & KPI SELECTION
    # =========================================================================
    unknown_campaigns = [
        campaign for campaign in selected_campaigns if campaign not in CAMPAIGN_SUMMARY_CSVS
    ]
    if unknown_campaigns:
        raise ValueError(f"Unknown campaign(s): {', '.join(unknown_campaigns)}")
    if not selected_campaigns:
        raise ValueError("Select at least one campaign.")

    if selected_kpis is None:
        kpis_to_plot = list(KPI_ORDER)
    else:
        unknown_kpis = [kpi for kpi in selected_kpis if kpi not in KPI_LABELS]
        if unknown_kpis:
            raise ValueError(f"Unknown KPI(s): {', '.join(unknown_kpis)}")
        if not selected_kpis:
            raise ValueError("Select at least one KPI.")
        kpis_to_plot = list(selected_kpis)

    n_kpis = len(kpis_to_plot)

    # =========================================================================
    # 2. SUBPLOT SETUP & SIZING
    # =========================================================================
    # Scale vertical spacing in pixel terms (~70px between subplots) so it remains constant regardless of row count
    total_height = height_per_kpi * n_kpis
    vertical_spacing = (70.0 / total_height) if n_kpis > 1 else 0.0

    fig = make_subplots(
        rows=n_kpis,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=vertical_spacing,
    )

    # =========================================================================
    # 3. Y-AXIS POSITIONING & OFFSETS
    # =========================================================================
    # Keep all operating modes on every panel, including modes with no source rows.
    mode_values = MODE_ORDER.copy()
    mode_positions = {mode: i * row_step for i, mode in enumerate(mode_values)}
    offsets = np.linspace(-0.30, 0.30, len(selected_campaigns))
    campaign_offsets = dict(zip(selected_campaigns, offsets))
    band_half_height = row_step / 2.0
    # Exact alignment with the top of Batch (i=0) and bottom of Continuous Perfusion (i=len-1)
    y_bounds = [-band_half_height, (len(mode_values) - 1) * row_step + band_half_height]

    for row_number, kpi in enumerate(kpis_to_plot, start=1):
        xref_domain = "x domain" if row_number == 1 else f"x{row_number} domain"
        yref_axis = "y" if row_number == 1 else f"y{row_number}"
        xaxis_name = "x" if row_number == 1 else f"x{row_number}"

        # =====================================================================
        # 4. BANDED BACKGROUND ROWS (OPERATIONAL MODES)
        # =====================================================================
        for i in range(len(mode_values)):
            if i % 2 == 0:
                y_center = i * row_step
                fig.add_shape(
                    type="rect",
                    x0=0,
                    x1=1,
                    xref=xref_domain,
                    y0=y_center - band_half_height,
                    y1=y_center + band_half_height,
                    yref=yref_axis,
                    fillcolor="rgba(0, 0, 0, 0.04)",
                    line_width=1,
                    line_color="rgba(0, 0, 0, 0.08)",
                    layer="below",
                )

        # =====================================================================
        # 5. CONVENTIONAL MEAT BENCHMARKS (DARK-TO-LIGHT GRADIENT & LINES)
        # =====================================================================
        # Determine data maximum for the currently selected campaigns in this subplot
        kpi_data = summary_all[summary_all["kpi"] == kpi]
        selected_kpi_data = kpi_data[kpi_data["campaign"].isin(selected_campaigns)]
        data_max_val = selected_kpi_data["max_value"].max() if not selected_kpi_data.empty else 1.0

        if show_reference_bands:
            kpi_benchmarks = {
                meat: MEAT_BENCHMARKS_PLOTTED[meat][kpi]
                for meat in MEAT_BENCHMARKS_PLOTTED
                if kpi in MEAT_BENCHMARKS_PLOTTED[meat] and MEAT_BENCHMARKS_PLOTTED[meat][kpi] is not None
            }
            if kpi_benchmarks:
                vals = list(kpi_benchmarks.values())
                min_ref, max_ref = min(vals), max(vals)

                # Upper x bound calculated dynamically from selected campaigns and benchmarks
                kpi_x_max = max(data_max_val, max_ref) * 1.08

                # Dark-to-light opacity range (dark/heavier shade on left, light/clear on right)
                alpha_dark = 0.2
                alpha_light = 0.0
                dark_color = f"rgba(0, 0, 0, {alpha_dark})"
                light_color = f"rgba(0, 0, 0, {alpha_light})"

                # 5a. Dark zone: left of lowest meat benchmark (< min_ref)
                if min_ref > 0:
                    fig.add_shape(
                        type="rect",
                        x0=0,
                        x1=min_ref,
                        xref=xaxis_name,
                        y0=y_bounds[0],
                        y1=y_bounds[1],
                        yref=yref_axis,
                        fillcolor=dark_color,
                        line_width=0,
                        layer="below",
                    )

                # 5b. Transition gradient zone: between lowest and highest meat benchmark [min_ref, max_ref]
                n_gradient_steps = 30
                if max_ref > min_ref:
                    x_steps = np.linspace(min_ref, max_ref, n_gradient_steps + 1)
                    for k in range(n_gradient_steps):
                        t = (k + 0.5) / n_gradient_steps
                        alpha_step = alpha_dark * (1.0 - t) + alpha_light * t
                        step_color = f"rgba(0, 0, 0, {alpha_step:.4f})"
                        fig.add_shape(
                            type="rect",
                            x0=x_steps[k],
                            x1=x_steps[k + 1],
                            xref=xaxis_name,
                            y0=y_bounds[0],
                            y1=y_bounds[1],
                            yref=yref_axis,
                            fillcolor=step_color,
                            line_width=0,
                            layer="below",
                        )

                # 5c. Light zone: right of highest meat benchmark (> max_ref)
                fig.add_shape(
                    type="rect",
                    x0=max_ref,
                    x1=kpi_x_max,
                    xref=xaxis_name,
                    y0=y_bounds[0],
                    y1=y_bounds[1],
                    yref=yref_axis,
                    fillcolor=light_color,
                    line_width=0,
                    layer="below",
                )

                # 5d. Vertical lines for individual meat types
                meat_line_styles = {
                    "Beef": dict(color="#8B1E0F", dash="dash"),
                    "Pork": dict(color="#D95F02", dash="dashdot"),
                    "Poultry": dict(color="#2E7D32", dash="dot"),
                }
                for meat, val in kpi_benchmarks.items():
                    raw_val = MEAT_BENCHMARKS_RAW[meat][kpi]
                    style = meat_line_styles.get(meat, dict(color="#555555", dash="dash"))
                    fig.add_trace(
                        go.Scatter(
                            x=[val, val],
                            y=y_bounds,
                            mode="lines",
                            line=dict(color=style["color"], width=4, dash=style["dash"]),
                            name=f"{meat} benchmark",
                            legendgroup=f"ref_{meat}",
                            showlegend=(row_number == 1),
                            hovertemplate=(
                                f"<b>{meat} reference</b><br>"
                                + f"Plotted: {val:.4g}<br>"
                                + f"Raw: {raw_val:.4g}<extra></extra>"
                            ),
                        ),
                        row=row_number,
                        col=1,
                    )
            else:
                kpi_x_max = data_max_val * 1.08
        else:
            kpi_x_max = data_max_val * 1.08

        # =====================================================================
        # 6. CAMPAIGN DATA: STEMS & MARKERS
        # =====================================================================
        for campaign in selected_campaigns:
            campaign_data = kpi_data[kpi_data["campaign"] == campaign].copy()
            if campaign_data.empty:
                continue

            y = campaign_data["mode_group"].astype(str).map(mode_positions).to_numpy(dtype=float)
            y = y + campaign_offsets[campaign]
            color = CAMPAIGN_COLORS[campaign]
            symbol = CAMPAIGN_SYMBOLS[campaign]
            label = CAMPAIGN_LABELS[campaign]

            # --- 6a. Range stems (min to max) ---
            for x_min, x_max, y_value, mode, n_runs in zip(
                campaign_data["min_value"],
                campaign_data["max_value"],
                y,
                campaign_data["mode_group"].astype(str),
                campaign_data["n_runs"],
            ):
                fig.add_trace(
                    go.Scatter(
                        x=[x_min, x_max],
                        y=[y_value, y_value],
                        mode="lines",
                        line=dict(color=color, width=4),
                        name=label,
                        legendgroup=campaign,
                        showlegend=False,
                        customdata=[[label, mode, n_runs], [label, mode, n_runs]],
                        hovertemplate=(
                            "<b>%{customdata[0]}</b><br>"
                            "Mode: %{customdata[1]}<br>"
                            "Range: %{x:.4g}<br>"
                            "Runs: %{customdata[2]}<extra></extra>"
                        ),
                    ),
                    row=row_number,
                    col=1,
                )

            # --- 6b. Min & max endpoint markers (hollow) ---
            endpoint_x = np.column_stack([campaign_data["min_value"], campaign_data["max_value"]]).ravel()
            endpoint_y = np.column_stack([y, y]).ravel()
            endpoint_customdata = np.column_stack(
                [
                    np.repeat(label, 2 * len(campaign_data)),
                    np.repeat(campaign_data["mode_group"].astype(str).to_numpy(), 2),
                    np.tile(["Min", "Max"], len(campaign_data)),
                    np.repeat(campaign_data["n_runs"].to_numpy(), 2),
                ]
            )
            fig.add_trace(
                go.Scatter(
                    x=endpoint_x,
                    y=endpoint_y,
                    mode="markers",
                    marker=dict(
                        size=12,
                        color="white",
                        symbol=symbol,
                        line=dict(color=color, width=2),
                    ),
                    name=f"{label} min/max",
                    legendgroup=campaign,
                    showlegend=False,
                    customdata=endpoint_customdata,
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Mode: %{customdata[1]}<br>"
                        "%{customdata[2]}: %{x:.4g}<br>"
                        "Runs: %{customdata[3]}<extra></extra>"
                    ),
                ),
                row=row_number,
                col=1,
            )

            # --- 6c. Mean markers (filled) ---
            fig.add_trace(
                go.Scatter(
                    x=campaign_data["mean_value"],
                    y=y,
                    mode="markers",
                    marker=dict(
                        size=8,
                        color=color,
                        symbol=symbol,
                        line=dict(color="black", width=2),
                    ),
                    name=label,
                    legendgroup=campaign,
                    showlegend=(row_number == 1),
                    customdata=np.column_stack(
                        [
                            np.full(len(campaign_data), label),
                            campaign_data["mode_group"].astype(str),
                            campaign_data["n_runs"],
                        ]
                    ),
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Mode: %{customdata[1]}<br>"
                        "Mean: %{x:.4g}<br>"
                        "Runs: %{customdata[2]}<extra></extra>"
                    ),
                ),
                row=row_number,
                col=1,
            )

        # =====================================================================
        # 7. SUBPLOT AXES STYLING
        # =====================================================================
        fig.update_yaxes(
            tickmode="array",
            tickvals=list(mode_positions.values()),
            ticktext=mode_values,
            range=y_bounds,
            autorange="reversed",
            showgrid=False,
            zeroline=False,
            tickfont=dict(size=18),
            row=row_number,
            col=1,
        )
        fig.update_xaxes(
            range=[0, kpi_x_max],
            showgrid=False,
            zeroline=False,
            tickfont=dict(size=18),
            row=row_number,
            col=1,
        )

    # =========================================================================
    # 8. GLOBAL LAYOUT, MARGINS & LEGEND
    # =========================================================================
    # Calculate legend gap and margins dynamically based on total figure height
    top_margin = 85
    bottom_margin = 75
    legend_y = 1.0 + (55.0 / total_height)

    fig.update_layout(
        template="plotly_white",
        height=total_height,
        width=width,
        margin=dict(l=210, r=40, t=top_margin, b=bottom_margin),
        font=dict(family="Arial, sans-serif", size=18, color="#222222"),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=legend_y,
            xanchor="left",
            x=0,
            font=dict(size=10),
        ),
    )

    # =========================================================================
    # 9. SUBPLOT TITLES & ANNOTATIONS
    # =========================================================================
    # Place each KPI title under its respective subplot using paper coordinates scaled to actual height
    title_gap_paper = 36.0 / total_height
    for row_number, kpi in enumerate(kpis_to_plot, start=1):
        yaxis_name = "yaxis" if row_number == 1 else f"yaxis{row_number}"
        panel_domain = fig.layout[yaxis_name].domain
        fig.add_annotation(
            text=KPI_LABELS[kpi],
            x=0.5,
            xref="paper",
            y=panel_domain[0] - title_gap_paper,
            yref="paper",
            xanchor="center",
            yanchor="top",
            showarrow=False,
            font=dict(size=18, color="#222222"),
        )

    return fig


# =============================================================================
# RUN FULL FIGURE
# =============================================================================
SHOW_REFERENCE_BANDS = True
fig = create_lollipop_figure(list(CAMPAIGN_SUMMARY_CSVS), show_reference_bands=SHOW_REFERENCE_BANDS)
fig.show()

# Responsive HTML export of the full figure
html_dir = CROSS_CAMPAIGN_DIR / "export" / "html"
html_dir.mkdir(parents=True, exist_ok=True)
full_html_path = html_dir / "all_campaigns_kpi_lollipop.html"
fig.write_html(
    str(full_html_path),
    include_plotlyjs="cdn",
    full_html=True,
    config={"displaylogo": False, "responsive": True},
)
print(f"Saved responsive interactive figure: {full_html_path}")

Saved responsive interactive figure: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\Operational Mode Screening\Cross_Campaign_Comparison\export\html\all_campaigns_kpi_lollipop.html


In [ ]:
# Configure campaigns and metrics for a focused comparison and optional export.
SELECTED_CAMPAIGNS = ["Campaign_2", "Campaign_4"]
# Optional: select a subset of metrics/KPIs (e.g. ["STY_WCM_kg_m3_day", "protein_conversion_efficiency_kg_kg"]), or None for all
KPI_TO_PLOT = [
    "STY_WCM_kg_m3_day",
    # "total_WCM_harvested_kg",
    "feed_conversion_efficiency_kg_kg_WCM",
    "protein_conversion_efficiency_kg_kg",
    "medium_conversion_efficiency_m3_kg_WCM",
]

SHOW_REFERENCE_BANDS = True
SAVEFIG = True
EXPORT_FORMAT = "svg"  # Supported by Plotly/Kaleido, e.g. "svg", "png", or "pdf".

comparison_fig = create_lollipop_figure(
    SELECTED_CAMPAIGNS,
    selected_kpis=KPI_TO_PLOT,
    height_per_kpi=300,
    show_reference_bands=SHOW_REFERENCE_BANDS,
)
comparison_fig.show()

if SAVEFIG:
    export_dir = CROSS_CAMPAIGN_DIR / "export" / "svg"
    html_dir = CROSS_CAMPAIGN_DIR / "export" / "html"
    html_dir.mkdir(parents=True, exist_ok=True)
    export_dir.mkdir(parents=True, exist_ok=True)
    campaign_suffix = "_vs_".join(campaign.lower() for campaign in SELECTED_CAMPAIGNS)
    output_path = export_dir / f"{campaign_suffix}_kpi_lollipop.{EXPORT_FORMAT}"
    # output_path = export_dir / f"{campaign_suffix}_lollipop_legend.{EXPORT_FORMAT}"

    # Export with transparent background
    export_fig = go.Figure(comparison_fig)
    export_fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    export_fig.write_image(str(output_path), format=EXPORT_FORMAT)
    print(f"Saved: {output_path}")

    # html_path = html_dir / f"{campaign_suffix}_kpi_lollipop.html"
    # comparison_fig.write_html(
    #     str(html_path),
    #     include_plotlyjs="cdn",
    #     full_html=True,
    #     config={"displaylogo": False, "responsive": True},
    # )
    # print(f"Saved interactive figure: {html_path}")

Saved: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\Operational Mode Screening\Cross_Campaign_Comparison\export\svg\campaign_2_vs_campaign_4_kpi_lollipop.svg
Saved interactive figure: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\Operational Mode Screening\Cross_Campaign_Comparison\export\html\campaign_2_vs_campaign_4_kpi_lollipop.html
